# Agoda DSA Master Jupyter Notebook

This notebook is a **pattern-first** revision guide for interview preparation.

It is designed to help you:
- recognize patterns quickly in a HackerRank / live coding round
- avoid common mistakes
- understand edge cases before coding
- practice a focused list of problems, including:
  - problems from your uploaded **real interview question document**
  - public Agoda interview themes from the last year

## How to use this notebook

For every question, ask in this order:
1. What is the input shape? array / string / matrix / tree / graph / linked list
2. What is being optimized? length / count / min / max / feasibility / shortest path / top K
3. Is it contiguous? → sliding window / prefix sum / Kadane / deque
4. Is it sorted or sortable? → two pointers / binary search / greedy / line sweep
5. Is there a repeated-state relationship? → DP
6. Does it ask for nearest greater/smaller? → monotonic stack
7. Does it involve dependencies or prerequisites? → graph / topo sort

## Agoda-focused themes to prioritize
Recent public Agoda reports and your uploaded notes repeatedly point to:
- two-question live HackerRank rounds with easy-medium to medium problems
- stock-buy/sell style DP/greedy variants
- Jump Game-like DP/greedy problems
- graph/tree medium problems
- binary-search-on-answer problems similar to Koko Eating Bananas
- min-difference / pairing / array optimization problems
- palindromic substring questions
- stacks / backtracking in some 2025 rounds
- some online assessments included **1 DSA + 1 HTTP/API basics**

# 1. Arrays: Core Patterns

Arrays are the highest-frequency bucket because they combine naturally with hashing, two pointers,
sliding window, prefix sum, sorting, greedy, and binary search.

## 1.1 Two Pointers

### What it is
Use two indices to avoid nested loops.

### When to use
- sorted arrays
- pair / triplet search
- in-place cleanup
- palindrome checks
- merge/intersection

### Variants
1. Opposite-direction pointers: `left = 0`, `right = n - 1`
2. Fast/slow pointers: `fast` scans, `slow` writes or tracks compressed result

### Core trick
You do **not** try every pair. You move one pointer based on a condition that narrows the search.

### Do's
- sort first if the logic depends on order
- write pointer movement rules before coding
- test duplicates and boundaries

### Don'ts
- do not move both pointers blindly
- do not forget `while left < right`
- do not lose original indices if sorting destroys them

### Common mistakes
- off-by-one on `left < right`
- forgetting duplicates in 3Sum / pair problems
- overwriting values too early in fast/slow pointer problems

In [ ]:
def two_sum_sorted(nums, target):
    left, right = 0, len(nums) - 1

    while left < right:
        s = nums[left] + nums[right]
        if s == target:
            return [left, right]
        elif s < target:
            left += 1
        else:
            right -= 1

    return []


def remove_duplicates_sorted(nums):
    if not nums:
        return 0

    slow = 1
    for fast in range(1, len(nums)):
        if nums[fast] != nums[fast - 1]:
            nums[slow] = nums[fast]
            slow += 1
    return slow

### Practice problems
- Two Sum II – Input Array Is Sorted
- Remove Duplicates from Sorted Array
- 3Sum
- Container With Most Water
- Valid Palindrome

### From your real interview notes
- Merge two sorted arrays
- Merge two sorted arrays **without extra space**
- Min absolute difference pairs / minimum sum pairs

## 1.2 Sliding Window

### What it is
Maintain a dynamic contiguous window `[left, right]`.

### When to use
- substring / subarray
- contiguous validity rule
- longest / shortest valid segment
- “at most K”
- no duplicate characters
- frequency-constrained windows

### Core idea
Expand `right`, and when the window becomes invalid, shrink `left` until it becomes valid again.

### Signals
- longest substring
- smallest subarray
- contiguous
- at most / at least / exactly

### Do's
- define the validity condition
- define what makes the window invalid
- use `while` to shrink, not `if`

### Don'ts
- do not check membership in `s[left:right+1]` inside the loop
- do not move `left` backward
- do not forget to update the best answer

### Common mistakes
- hidden `O(n^2)` due to slicing or repeated counting
- stale indices in hashmap version
- forgetting `char_index[ch] >= left`

In [ ]:
def longest_unique_length(s):
    char_index = {}
    left = 0
    best = 0

    for right, ch in enumerate(s):
        if ch in char_index and char_index[ch] >= left:
            left = char_index[ch] + 1

        char_index[ch] = right
        best = max(best, right - left + 1)

    return best

### Edge cases
- empty string
- one character
- all same characters
- all unique characters
- tricky stale-index case like `"abba"`

### Practice problems
- Longest Substring Without Repeating Characters
- Minimum Window Substring
- Longest Repeating Character Replacement
- Permutation in String
- Subarrays with K Different Integers

### From your real interview notes
- Longest substring without repeating characters

## 1.3 Prefix Sum

### What it is
Prefix sum = cumulative sum up to current index.

### When to use
- subarray sum equals K
- longest zero-sum subarray
- equal 0s and 1s
- range-sum queries
- counting subarrays with a target sum

### Core identity
If `prefix[j] == prefix[i]`, then the subarray `(i+1 ... j)` sums to zero.

### Key base case
Use `{0: -1}`.
This means:
- before the array starts, prefix sum is 0 at a virtual index `-1`
- it lets you detect valid subarrays that start at index `0`

### Do's
- store the **first occurrence** when looking for longest length
- store counts when the question asks "how many subarrays"
- convert 0 to -1 for equal-0-and-1 style problems

### Don'ts
- do not compare indices directly when length is what matters
- do not forget off-by-one: start is `first_seen + 1`

### Common mistakes
- forgetting `{0: -1}`
- not tracking `max_len`
- using wrong start index
- mixing up "first seen" vs "count of seen"

In [ ]:
def longest_zero_sum(nums):
    prefix = 0
    first_seen = {0: -1}
    best = 0

    for i, x in enumerate(nums):
        prefix += x
        if prefix in first_seen:
            best = max(best, i - first_seen[prefix])
        else:
            first_seen[prefix] = i

    return best

### Practice problems
- Subarray Sum Equals K
- Contiguous Array
- Maximum Size Subarray Sum Equals k
- Range Sum Query – Immutable
- Find Pivot Index

### From your real interview notes
- Binary array with equal 0s and 1s

## 1.4 Sorting Pattern

### What it is
Sort first to make the structure easier to exploit.

### When to use
- intervals
- greedy scheduling
- grouping duplicates
- pairing and merging
- nearest-difference problems

### Core trick
Sorting often converts a hard problem into a simple scan.

### Common mistakes
- wrong sort key
- missing tie-breakers
- forgetting that sorting changes adjacency

In [ ]:
def merge_intervals(intervals):
    intervals.sort(key=lambda x: x[0])
    merged = []

    for start, end in intervals:
        if not merged or merged[-1][1] < start:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)

    return merged

### Practice problems
- Merge Intervals
- Insert Interval
- Non-overlapping Intervals
- Meeting Rooms
- Minimum Number of Arrows to Burst Balloons

### From your real interview notes
- minimum sum pairs
- min absolute difference = K style pairs

## 1.5 Subarrays Pattern

### Tools that appear repeatedly
- sliding window
- prefix sum
- Kadane’s algorithm
- hashing
- deque

### Pattern selection
- max sum contiguous subarray → Kadane
- sum condition → prefix sum
- longest valid under local constraint → sliding window
- max/min in every window → deque

### Practice problems
- Maximum Subarray
- Subarray Sum Equals K
- Maximum Product Subarray
- Shortest Subarray with Sum at Least K
- Contiguous Array

# 2. Stacks and Queues

These patterns are common because they reveal whether you can:
- exploit order
- maintain a monotonic invariant
- simulate structure efficiently
- handle nested structure and parsing

## 2.1 Monotonic Stack

### What it is
A stack maintained in increasing or decreasing order.

### When to use
- next greater / smaller
- previous greater / smaller
- nearest element queries
- histogram rectangle
- stock span / temperature style questions

### Signals
- for each element, find the next/previous ...
- nearest greater/smaller
- how many until a bigger value
- largest rectangle / boundaries

### Common mistakes
- wrong comparison sign (`<` vs `<=`)
- forgetting stale indices in window/deque variants
- not understanding whether equal values should be popped

In [ ]:
def next_greater(nums):
    result = [-1] * len(nums)
    stack = []  # indices

    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            idx = stack.pop()
            result[idx] = x
        stack.append(i)

    return result

### Practice problems
- Next Greater Element I
- Daily Temperatures
- Online Stock Span
- Largest Rectangle in Histogram
- Trapping Rain Water

## 2.2 Min Stack / Max Stack

### What it is
A stack that supports retrieving current min or max in `O(1)`.

### Core trick
Store the running min/max together with the value.

In [ ]:
class MinStack:
    def __init__(self):
        self.stack = []

    def push(self, val):
        curr_min = val if not self.stack else min(val, self.stack[-1][1])
        self.stack.append((val, curr_min))

    def pop(self):
        return self.stack.pop()[0]

    def top(self):
        return self.stack[-1][0]

    def getMin(self):
        return self.stack[-1][1]

### Practice problems
- Min Stack
- Max Stack
- Design Browser History
- Design a Stack With Increment Operation
- Flatten Nested List Iterator

## 2.3 Parenthesis Stack Pattern

### What it is
Use stack to validate, clean, or decode nested structure.

### When to use
- balanced parentheses
- remove invalid parentheses
- decode nested strings
- evaluate nested expressions

### Common mistakes
- not handling empty stack
- not validating at the end
- confusing generation with validation

In [ ]:
def is_valid_parentheses(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = []

    for ch in s:
        if ch in '([{':
            stack.append(ch)
        else:
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()

    return not stack

### Practice problems
- Valid Parentheses
- Minimum Remove to Make Valid Parentheses
- Generate Parentheses
- Decode String
- Basic Calculator II

## 2.4 Queue / Deque Pattern

### What it is
Use `deque` for efficient push/pop from both ends.

### When to use
- BFS
- sliding window max/min
- circular queue/deque
- monotonic queue
- stream-like processing

### Common mistakes
- using list pop(0) as a queue
- not removing out-of-window indices
- leaving expired indices in the deque

In [ ]:
from collections import deque

def max_sliding_window(nums, k):
    dq = deque()
    ans = []

    for i, x in enumerate(nums):
        while dq and dq[0] <= i - k:
            dq.popleft()

        while dq and nums[dq[-1]] <= x:
            dq.pop()

        dq.append(i)

        if i >= k - 1:
            ans.append(nums[dq[0]])

    return ans

### Practice problems
- Sliding Window Maximum
- Design Circular Queue
- Design Circular Deque
- Rotting Oranges
- Shortest Subarray with Sum at Least K

# 3. Trees

Tree interviews test whether you can:
- define recursion cleanly
- distinguish local vs global state
- choose BFS vs DFS correctly
- use BST ordering when available

## 3.1 Tree DFS

### What it is
Depth-first traversal through recursion or an explicit stack.

### When to use
- subtree computations
- height/depth
- path-sum logic
- validation
- structure transforms

### Core trick
Define what the recursive function returns.

### Common mistakes
- unclear base case
- mixing return value with global answer
- path-local state bugs

In [ ]:
def max_depth(root):
    if not root:
        return 0
    return 1 + max(max_depth(root.left), max_depth(root.right))

### Practice problems
- Maximum Depth of Binary Tree
- Path Sum
- Balanced Binary Tree
- Binary Tree Paths
- Invert Binary Tree

## 3.2 Tree BFS

### What it is
Level-order traversal using a queue.

### When to use
- level-order output
- nearest-level relation
- zigzag / right side view / level aggregates

In [ ]:
from collections import deque

def level_order(root):
    if not root:
        return []

    q = deque([root])
    ans = []

    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        ans.append(level)

    return ans

### Practice problems
- Binary Tree Level Order Traversal
- Binary Tree Right Side View
- Average of Levels in Binary Tree
- Zigzag Level Order Traversal
- Minimum Depth of Binary Tree

## 3.3 BST Pattern

### What it is
Exploit the BST ordering: left < root < right.

### Common mistakes
- validating only parent-child relation instead of full allowable range
- forgetting duplicates policy

In [ ]:
def search_bst(root, val):
    while root:
        if root.val == val:
            return root
        elif val < root.val:
            root = root.left
        else:
            root = root.right
    return None

### Practice problems
- Search in a Binary Search Tree
- Validate Binary Search Tree
- Kth Smallest Element in a BST
- Lowest Common Ancestor of a BST
- Convert Sorted Array to BST

## 3.4 Diameter / Tree DP Pattern

### What it is
Compute a local quantity at each node while updating a global best.

### Practice problems
- Diameter of Binary Tree
- Binary Tree Maximum Path Sum
- Longest Univalue Path
- House Robber III
- Count Good Nodes in Binary Tree

## 3.5 LCA Pattern

### What it is
Lowest Common Ancestor of two nodes.

### Core idea
A node is LCA if:
- one target is in one subtree and the other target is in the other subtree
- or the node itself is one of the targets

### Practice problems
- Lowest Common Ancestor of a Binary Tree
- Lowest Common Ancestor of a BST
- Smallest Common Region
- Step-By-Step Directions From a Binary Tree Node to Another
- Distance Between Two Nodes in BST

### From your real interview notes
- LCA of a tree given key and level at each node

# 4. Graphs

Graph problems are often just trees/matrices/general state spaces with a different representation.
Main buckets:
- BFS
- DFS
- topological sort
- shortest path
- cycle detection
- components

## 4.1 Graph BFS

### What it is
Breadth-first traversal on graph states.

### When to use
- shortest path in unweighted graph
- minimum steps
- layer-by-layer expansion
- multi-source spread

### Common mistakes
- marking visited too late
- using DFS for shortest unweighted path
- forgetting multi-source initialization

In [ ]:
from collections import deque, defaultdict

def bfs(graph, start):
    q = deque([start])
    visited = {start}

    while q:
        node = q.popleft()
        for nei in graph[node]:
            if nei not in visited:
                visited.add(nei)
                q.append(nei)

### Practice problems
- Word Ladder
- Rotting Oranges
- Open the Lock
- Minimum Genetic Mutation
- Shortest Path in Binary Matrix

## 4.2 Graph DFS

### What it is
Traverse deeply through neighbors using recursion or stack.

### When to use
- connected components
- island counting
- reachability
- cycle detection
- graph cloning

### Practice problems
- Number of Islands
- Clone Graph
- Pacific Atlantic Water Flow
- Reconstruct Itinerary
- Keys and Rooms

## 4.3 Topological Sort

### What it is
Ordering nodes of a DAG by dependency.

### Two standard ways
1. Kahn’s algorithm (BFS with indegree)
2. DFS with recursion stack / postorder

### Common mistakes
- not detecting cycles
- forgetting topo sort only works on DAGs

In [ ]:
from collections import defaultdict, deque

def topo_sort(n, edges):
    graph = defaultdict(list)
    indeg = [0] * n

    for u, v in edges:
        graph[u].append(v)
        indeg[v] += 1

    q = deque([i for i in range(n) if indeg[i] == 0])
    order = []

    while q:
        node = q.popleft()
        order.append(node)
        for nei in graph[node]:
            indeg[nei] -= 1
            if indeg[nei] == 0:
                q.append(nei)

    return order if len(order) == n else []

### Practice problems
- Course Schedule
- Course Schedule II
- Alien Dictionary
- Parallel Courses
- Sequence Reconstruction

## 4.4 Shortest Path

### Decision rule
- unweighted graph → BFS
- weighted graph, non-negative weights → Dijkstra
- negative weights → Bellman-Ford / specialized methods

### Common mistakes
- using BFS for weighted graph
- not skipping stale heap entries
- mixing edge count with edge weight

In [ ]:
import heapq
from collections import defaultdict

def dijkstra(n, edges, src):
    graph = defaultdict(list)
    for u, v, w in edges:
        graph[u].append((v, w))

    dist = {i: float('inf') for i in range(n)}
    dist[src] = 0
    heap = [(0, src)]

    while heap:
        d, node = heapq.heappop(heap)
        if d > dist[node]:
            continue

        for nei, w in graph[node]:
            nd = d + w
            if nd < dist[nei]:
                dist[nei] = nd
                heapq.heappush(heap, (nd, nei))

    return dist

### Practice problems
- Network Delay Time
- Cheapest Flights Within K Stops
- Path With Minimum Effort
- Swim in Rising Water
- Minimum Cost to Make at Least One Valid Path in a Grid

## 4.5 Cycle Detection

### Undirected graph
Use DFS/BFS with parent tracking, or Union-Find.

### Directed graph
Use DFS with recursion stack, or detect failure of topo sort.

### Practice problems
- Graph Valid Tree
- Course Schedule
- Redundant Connection
- Redundant Connection II
- Detect Cycle in Directed Graph

## 4.6 Connected Components

### What it is
Every unvisited node starts a new component.

### Practice problems
- Number of Connected Components in an Undirected Graph
- Number of Provinces
- Number of Islands
- Accounts Merge
- Surrounded Regions

# 5. Dynamic Programming

DP is not one pattern; it is a family of patterns.

## The 4 mandatory DP questions before coding
1. What is the state?
2. What is the transition?
3. What are the base cases?
4. Can memory be compressed?

## 5.1 1D DP

### Signals
- number of ways
- min cost / max profit over a line
- climb / jump / rob style

### Common mistakes
- wrong base cases
- forcing DP when greedy works
- not simplifying to rolling variables

In [ ]:
def climb_stairs(n):
    if n <= 2:
        return n

    a, b = 1, 2
    for _ in range(3, n + 1):
        a, b = b, a + b
    return b

### Practice problems
- Climbing Stairs
- House Robber
- Min Cost Climbing Stairs
- Decode Ways
- Maximum Product Subarray

### From your real interview notes
- staircase count ways

## 5.2 2D DP

### Signals
- matrix path optimization
- edit distance / LCS style
- choose/skip across two sequences

### Common mistakes
- wrong table dimensions
- wrong base row/column
- mixing 0-index and 1-index transitions

In [ ]:
def unique_paths(m, n):
    dp = [[1] * n for _ in range(m)]
    for r in range(1, m):
        for c in range(1, n):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
    return dp[-1][-1]

### Practice problems
- Unique Paths
- Minimum Path Sum
- Longest Common Subsequence
- Edit Distance
- Distinct Subsequences

## 5.3 DP Compression

### What it is
Reduce memory when only previous row/state is needed.

### Examples
- Fibonacci / climbing stairs → 2 variables
- 2D grid DP → one row
- knapsack → one dimension with reverse iteration

### Practice problems
- Climbing Stairs
- Min Cost Climbing Stairs
- Unique Paths
- Triangle
- Maximal Square

## 5.4 Subsequence DP

### Signals
- choose or skip
- order matters but contiguity does not
- compare/match two sequences

### Practice problems
- Longest Common Subsequence
- Distinct Subsequences
- Longest Increasing Subsequence
- Edit Distance
- Palindromic Subsequences

## 5.5 Knapsack Pattern

### Core idea
For each item:
- take it
- skip it

### Common mistakes
- wrong loop order in 1D compression
- not distinguishing 0/1 from unlimited use
- wrong base capacity handling

In [ ]:
def can_partition(nums):
    total = sum(nums)
    if total % 2:
        return False

    target = total // 2
    dp = [False] * (target + 1)
    dp[0] = True

    for x in nums:
        for s in range(target, x - 1, -1):
            dp[s] = dp[s] or dp[s - x]

    return dp[target]

### Practice problems
- Partition Equal Subset Sum
- Target Sum
- Coin Change
- Coin Change II
- 0/1 Knapsack

## 5.6 Tree DP

### What it is
Dynamic programming where each node returns some state upward.

### Practice problems
- House Robber III
- Binary Tree Maximum Path Sum
- Diameter of Binary Tree
- Longest ZigZag Path in a Binary Tree
- Distribute Coins in Binary Tree

## Agoda/public last-year DP-style themes
- stock buy/sell variation
- Jump Game-like DP
- count palindromic substrings

# 6. Trie

### What it is
A prefix tree for strings.

### When to use
- autocomplete
- prefix search
- dictionary with wildcard search
- replace words
- multiple-word board search

### Common mistakes
- forgetting end-of-word marker
- using trie where hashing is enough
- not thinking about memory cost

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True

    def search(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return node.is_end

    def startsWith(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return True

### Practice problems
- Implement Trie (Prefix Tree)
- Design Add and Search Words Data Structure
- Replace Words
- Word Search II
- Search Suggestions System

### From your real interview notes
- Implement Autocomplete using Trie

# 7. Hashing

### What it is
Use hash map / hash set for average `O(1)` lookup.

### When to use
- frequency counting
- duplicate detection
- complement lookup
- grouping
- visited tracking

### Common mistakes
- choosing set when map is needed
- not differentiating first seen vs count seen
- mutating values before using them

### Practice problems
- Two Sum
- Group Anagrams
- Valid Anagram
- Longest Consecutive Sequence
- Top K Frequent Elements

# 8. Matrix Traversal

### What it is
Grid BFS/DFS or directional simulation.

### Typical directions

In [ ]:
DIRS_4 = [(1,0), (-1,0), (0,1), (0,-1)]
DIRS_8 = [(1,0), (-1,0), (0,1), (0,-1), (1,1), (1,-1), (-1,1), (-1,-1)]

### Common mistakes
- row/column boundary bugs
- revisiting cells
- wrong visited timing
- using 4-neighbor logic when 8-neighbor is needed

### Practice problems
- Number of Islands
- Flood Fill
- Rotting Oranges
- Spiral Matrix
- Shortest Path in Binary Matrix

# 9. Line Sweep

### What it is
Process interval events in sorted order.

### When to use
- maximum overlap
- meeting rooms
- skyline
- event counting across time
- booking calendars

### Common mistakes
- wrong start/end tie-breaking
- not understanding open vs closed intervals
- using expensive interval comparisons instead of events

In [ ]:
def max_overlap(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 1))
        events.append((e, -1))

    events.sort()
    active = 0
    best = 0

    for _, delta in events:
        active += delta
        best = max(best, active)

    return best

### Practice problems
- Meeting Rooms II
- Employee Free Time
- The Skyline Problem
- Car Pooling
- My Calendar I / II

# 10. Heap

### What it is
Priority queue for repeated min/max extraction.

### When to use
- top K
- streaming median
- scheduling
- shortest path
- merging sorted streams

### Common mistakes
- sorting whole input when heap is more natural
- not limiting heap size in top-K problems
- stale heap entries in graph problems

In [ ]:
import heapq

def kth_largest(nums, k):
    heap = []
    for x in nums:
        heapq.heappush(heap, x)
        if len(heap) > k:
            heapq.heappop(heap)
    return heap[0]

### Practice problems
- Kth Largest Element in an Array
- Top K Frequent Elements
- Merge K Sorted Lists
- Find Median from Data Stream
- Task Scheduler

# 11. Linked List

### High-yield patterns
- reverse list
- merge sorted lists
- fast/slow pointers
- cycle detection
- dummy node technique

### Common mistakes
- losing `next` before rewiring
- mishandling head changes
- forgetting dummy node when deleting near head

In [ ]:
def reverse_list(head):
    prev = None
    curr = head

    while curr:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt

    return prev

### Practice problems
- Reverse Linked List
- Merge Two Sorted Lists
- Linked List Cycle
- Remove Nth Node From End of List
- Reorder List

# 12. Kadane’s Algorithm

### What it is
Pattern for maximum sum contiguous subarray.

### Core idea
At each element:
- either extend the previous best-ending-here subarray
- or start fresh from the current element

### Common mistakes
- wrong initialization on all-negative arrays
- confusing subarray with subsequence
- not understanding local best vs global best

In [ ]:
def max_subarray(nums):
    curr = best = nums[0]
    for x in nums[1:]:
        curr = max(x, curr + x)
        best = max(best, curr)
    return best

### Practice problems
- Maximum Subarray
- Maximum Sum Circular Subarray
- Maximum Product Subarray
- Best Time to Buy and Sell Stock
- Maximum Absolute Sum of Any Subarray

# 13. Binary Search on Answer

This is one of the most important patterns for Agoda-style interviews.

### What it is
Binary search not over an array index, but over a **feasible answer range**.

### When to use
- minimum possible X
- maximum feasible X
- can write a `feasible(mid)` function
- feasibility is monotonic

### Common mistakes
- wrong search range
- feasibility function not monotonic
- wrong final return
- saying `log n` when it is really `log(value_range)`

In [ ]:
def binary_search_answer(low, high, feasible):
    while low <= high:
        mid = (low + high) // 2
        if feasible(mid):
            high = mid - 1
        else:
            low = mid + 1
    return low

### Practice problems
- Koko Eating Bananas
- Capacity To Ship Packages Within D Days
- Split Array Largest Sum
- Minimum Number of Days to Make m Bouquets
- aggressive-cows / magnetic-force style placement

### From your real interview notes
- Minimum days to make bouquets

### From recent Agoda public reports
- binary-search problem similar to Koko Eating Bananas

# 14. Greedy

Greedy appears a lot in practical interviews because it tests whether you can identify the correct invariant instead of brute-forcing all possibilities.

### When to use
- interval scheduling
- jump/reachability
- stock variants
- placement / coverage
- merging and scanning

### Do's
- state the invariant
- explain why the local choice is safe
- test a counterexample mentally

### Don'ts
- do not assume greedy without a reason
- do not use DP if a clean invariant solves it

### High-yield Agoda-style greedy problems
- Jump Game
- Jump Game II
- Gas Station
- Non-overlapping Intervals
- Best Time to Buy and Sell Stock

### From your real interview notes
- Jump Game

### From recent Agoda public reports
- stock buy/sell variation
- Jump Game-like DP/greedy problem

# 15. Palindrome / String Expansion Pattern

This matters because it appears in your notes and recent Agoda public reports.

### Core methods
1. Expand Around Center
2. DP on substring ranges
3. Manacher’s algorithm (rarely needed in interviews)

### When to use
- longest palindromic substring
- count palindromic substrings

### Common mistakes
- forgetting even-length centers
- returning count when asked for substring
- off-by-one while expanding

### Practice problems
- Longest Palindromic Substring
- Palindromic Substrings
- Valid Palindrome II
- Longest Palindrome by Concatenating Two Letter Words
- Shortest Palindrome

### From your real interview notes
- longest palindromic substring

### From recent Agoda public reports
- count palindromic substrings

# 16. Special Problems From Your Real Interview Document

## Array / Greedy / Binary Search
- Minimum number of days to make bouquets
- Jump Game
- Stairs / climbing ways
- Missing number from 1 to n
- Next permutation
- Merge two sorted arrays
- Merge two sorted arrays without extra space
- Pair problems: min absolute difference / min sum pairs

## String / Hashing / Sliding Window
- Longest substring without repeating characters
- Longest palindromic substring
- Binary array with equal number of 0s and 1s
- Balanced-parenthesis style questions

## Tree / Graph
- LCA
- BFS on matrix / shortest path style

## Trie / Design
- Autocomplete using Trie

## ML-coded but still useful for implementation discipline
- TF-IDF
- matrix factorization
- attention with NumPy
- isolation forest
- gradient boosting iteration

# 17. Drill List by Pattern (3–5 each)

## Two pointers
- Two Sum II
- Remove Duplicates from Sorted Array
- 3Sum
- Container With Most Water
- Merge Two Sorted Arrays

## Sliding window
- Longest Substring Without Repeating Characters
- Minimum Window Substring
- Longest Repeating Character Replacement
- Permutation in String
- Subarrays with K Different Integers

## Prefix sum
- Subarray Sum Equals K
- Contiguous Array
- Maximum Size Subarray Sum Equals k
- Range Sum Query – Immutable
- Pivot Index

## Monotonic stack
- Daily Temperatures
- Next Greater Element I
- Largest Rectangle in Histogram
- Trapping Rain Water
- Sum of Subarray Minimums

## Parenthesis / stack
- Valid Parentheses
- Minimum Remove to Make Valid Parentheses
- Decode String
- Generate Parentheses
- Basic Calculator II

## Queue / deque
- Sliding Window Maximum
- Design Circular Queue
- Design Circular Deque
- Rotting Oranges
- Shortest Subarray with Sum at Least K

## Trees
- Maximum Depth of Binary Tree
- Binary Tree Level Order Traversal
- Validate Binary Search Tree
- Diameter of Binary Tree
- Lowest Common Ancestor of a Binary Tree

## Graphs
- Number of Islands
- Course Schedule
- Rotting Oranges
- Network Delay Time
- Number of Provinces

## DP
- Climbing Stairs
- House Robber
- Unique Paths
- Longest Common Subsequence
- Partition Equal Subset Sum

## Trie
- Implement Trie
- Add and Search Word
- Replace Words
- Word Search II
- Search Suggestions System

## Heap
- Kth Largest Element in an Array
- Top K Frequent Elements
- Merge K Sorted Lists
- Find Median from Data Stream
- Task Scheduler

## Linked list
- Reverse Linked List
- Merge Two Sorted Lists
- Linked List Cycle
- Remove Nth Node From End of List
- Reorder List

## Kadane
- Maximum Subarray
- Maximum Sum Circular Subarray
- Maximum Product Subarray
- Best Time to Buy and Sell Stock
- Maximum Absolute Sum of Any Subarray

## Binary search on answer
- Koko Eating Bananas
- Capacity To Ship Packages Within D Days
- Split Array Largest Sum
- Minimum Days to Make m Bouquets
- aggressive-cows / magnetic-force style

## Agoda/public last-year style extras
- stock buy/sell variant
- Jump Game-like minimum jumps / reachability
- graph/tree medium
- binary search like Koko
- min-difference pair problem
- count palindromic substrings
- stack/backtracking medium

# 18. Final Do's and Don'ts

## Do's
- identify the pattern before writing code
- say brute force first, then optimized approach
- test empty input, single element, duplicates, extremes
- keep helper functions small, especially feasibility checks
- mention time and space complexity after coding

## Don'ts
- do not hide `O(n^2)` inside a loop with slicing or repeated counts
- do not forget stale-index checks in hashmap window problems
- do not use DFS when BFS is required for shortest unweighted path
- do not use full DP table if a rolling state works
- do not over-engineer when a simple scan or greedy invariant is enough

# 19. Personal Practice Tracker

## Patterns I am strongest in
- 
- 
- 

## Patterns I still confuse
- 
- 
- 

## Mistakes I keep repeating
- 
- 
- 

## Problems to redo without help
- 
- 
- 

## Mock-interview targets
- finish first question in 20–25 minutes
- explain pattern before coding
- always state edge cases
- do not get stuck longer than 8–10 minutes without simplifying